In [ ]:
"""
session_summary.ipynb

plot model metrics across sessions

Author: Stellina X. Ao
Created: 2026-05-04
Last Modified: 2026-05-04
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt
from utils.paths import FIGURES_DIR

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

## LVM - one latent

In [ ]:
"""
ALL & PER-REGION
R2 of encoding and latent variable model 
Improvement over task variable model and QI
Single latent histograms
Quantify time scale of latents (PSD)
"""

In [ ]:
from utils.paths import MODELS_DIR
import pickle
import numpy as np
from core.data import subject_ids, session_ids, colors_subj


def get_res_tv_lvms(subj_id, regions=["all", "ACC", "M2", "DMS", "DLS"]):
    subj_idx = np.where(subject_ids == subj_id)[0][0]

    res_tv_lvms = []
    for sess_id in session_ids[subj_idx]:
        file_path = (
            MODELS_DIR / "fit" / subj_id / sess_id / "one_latent" / "results_dict.pkl"
        )

        with open(file_path, "rb") as f:
            res_dict = pickle.load(f)

        res_tv_lvm = {}
        for region in regions:
            try:
                res_tv_lvms_ = res_dict[region]["res_tv_lvms"]
            except KeyError:
                continue
            # for now
            if len(res_tv_lvms_) == 0:
                continue
            # print(subj_id, sess_id, region)
            res_tv_lvm[region] = {
                key: np.array([res_tv_lvm_[key] for res_tv_lvm_ in res_tv_lvms_])
                for key in res_tv_lvms_[0].keys()
            }
        res_tv_lvms.append(res_tv_lvm)

    return res_tv_lvms

In [ ]:
res_tv_lvms = {
    subj_id: get_res_tv_lvms(subj_id) for subj_id in ["MM012", "MR82", "MR83"]
}

In [ ]:
region = "ACC"
metric = "r2test_diff"

fig, ax = plt.subplots()
for subj_id in ["MM012", "MR82", "MR83"]:
    metrics_avg = np.array(
        [
            res_tv_lvm[region][metric].mean() if region in res_tv_lvm.keys() else np.nan
            for res_tv_lvm in res_tv_lvms[subj_id]
        ]
    )
    metrics_std = np.array(
        [
            res_tv_lvm[region][metric].std() if region in res_tv_lvm.keys() else np.nan
            for res_tv_lvm in res_tv_lvms[subj_id]
        ]
    )

    ax.plot(metrics_avg, color=colors_subj[subj_id], label=subj_id)
    ax.fill_between(
        np.arange(len(metrics_avg)),
        metrics_avg - metrics_std,
        metrics_avg + metrics_std,
        color=colors_subj[subj_id],
        alpha=0.25,
    )

ax.axhline(y=0, color="#666666", linestyle="--")
ax.set_xlabel("session")
ax.set_ylabel(metric)
ax.legend()
fig.tight_layout()


save_dir = FIGURES_DIR / "session_summary"
fpath = save_dir / f"{metric}_{region}.png"
fpath.parent.mkdir(parents=True, exist_ok=True)

fig.savefig(fpath, dpi=300, bbox_inches="tight")